In [1]:
import os
import nirjas
import pandas as pd
import numpy as np
from fuzzywuzzy import fuzz

In [42]:
class SemanticSearchAgent():
    """
    An agent that performs semantic search to identify potential licenses within files.
    """

    def __init__(self):
        """
        Initializes the SemanticSearchAgent with pre-computed license embeddings and a sentence embedding model.
        """
        
        self.license_dataset_df = pd.read_csv('extras/license_information/license_dataset.csv')
        
    def extract_comments(self, filePath: str):
        """
        Extracts comments from a file using the 'nirjas' library, falling back to reading the entire file if comment extraction fails.
        """
        if not os.path.exists(filePath):
           raise Exception(f"File path '{filePath}' does not exist")
        try: 
            # * Manually extracting comments using nirjas and not the commentPreprocessor class
            # * Because I prefer to do my own preprocessing and I also need accurate comment reading,
            # * Normal nirjas words by appending comments together.
            nirjas_comments = nirjas.extract(filePath)
            # if nirjas_comments.total_lines_of_comments == 0:
                # Go to the except case to read all the file, even if it has no comments
                # ! This is debatable, and I might remove it.
                # raise Exception()
            all_comments = []
            # Go through each comment type, and read the comment itself given it's starting and ending lines
            # This is necessary because nirjas by default appends multi-line or continous single-line comments
            # together, and for semantic search purposes, I want to read the comments exactly as they were in the file.
            with open(filePath, "r") as f:
                all_lines = f.readlines()
            for single_line_comment in nirjas_comments['single_line_comment']:
                all_comments.append(single_line_comment['comment'])
            for cont_single_line_comment in nirjas_comments['cont_single_line_comment']:
                start = cont_single_line_comment['start_line'] - 1
                end = cont_single_line_comment['end_line']
                for line_idx in range(start, end):
                    comment = all_lines[line_idx]
                    all_comments.append(comment)
            for multi_line_comment in nirjas_comments['multi_line_comment']:
                start = multi_line_comment['start_line'] - 1
                end = multi_line_comment['end_line']
                for line_idx in range(start, end):
                    line = all_lines[line_idx]
                    all_comments.append(line) 
            comments = "".join(all_comments)
        except:
            with open(filePath, "r") as f:
                comments = f.read()
        return comments

    # def scan(self, filePath): 
    #     '''
    #     Scans a file for potential licenses using semantic search and fuzzy string matching.

    #     :param processedData: Processed Data form input file
    #     :return: Returns possible licenses contained in the input file based
    #             on matching using semantic search
    #     '''

    #     file_comments = self.extract_comments(filePath)

    #     # Cleans up the extracted comments. Specifically remove these characters because they don't really show up in 
    #     # license texts. Optional, and can be removed without much issue (maybe a small issue for one or two licenses)
    #     chars_to_remove = ['—', '…', '•', '§', '«', '»', '„', '・', '−', '*', '>', '<']
    #     for char_to_remove in chars_to_remove:
    #         file_comments = file_comments.replace(char_to_remove, '')

    #     file_comments = file_comments.split('\n')

    #     # This map is used to match each line in a license text (in a file with all license texts merged together)
    #     # to its correct license index and thereby allow me to find out which license that line belongs to
    #     license_index_map = {}
    #     all_license_texts = []
    #     for license_index, license_text in enumerate(self.license_dataset_df['License Text']):
    #         for line in license_text.split('\n'):
    #             all_license_texts.append(line)
    #             license_index_map[len(all_license_texts) - 1] = license_index

    #     # semantic search using fuzzywuzzy, which uses levenshtein distance
    #     appended_comments = []
    #     results = []
    #     fuzzy_similarity_matrix = np.zeros((len(file_comments), len(all_license_texts)))
    #     for index, comment in enumerate(file_comments):
    #         file_comment = comment
    #         for i in range(len(all_license_texts)):
    #             fuzzy_similarity_matrix[index][i] = fuzz.ratio(file_comment, all_license_texts[i]) 
    #         max_score_index = np.argmax(fuzzy_similarity_matrix[index]) 
    #         license_index = license_index_map[max_score_index]
    #         results.append(
    #                         (
    #                             fuzzy_similarity_matrix[index][max_score_index], 
    #                             comment, self.license_dataset_df.loc[license_index, 'License Name'],
    #                             self.license_dataset_df.loc[license_index, 'License ID'],
    #                             all_license_texts[max_score_index],
    #                         )
    #                     )
        
    #     for result in results:
    #         if result[0] >= 30:
    #             appended_comments.append(result[1])
    #         else:
    #             break
        
    #     appended_comments = "\n".join(appended_comments)
    #     appended_comments_embeddings = self.model.encode(appended_comments).reshape(1, -1)
    #     fuzzy_similarity_matrix_2 = np.zeros(len(self.license_embeddings))
    #     fuzzy_similarity_matrix_2 = cosine_similarity(appended_comments_embeddings, self.license_embeddings)
    #     # for i in range(len(self.license_dataset_df)):
    #         # fuzzy_similarity_matrix_2[i] = fuzz.ratio(appended_comments, self.license_dataset_df.loc[i, 'License Text'])
    #         # t = cosine_similarity(appended_comments_embeddings, self.license_embeddings[i].reshape(1, -1))
    #         # fuzzy_similarity_matrix_2[i] = t
    #     top_5_indices = np.argsort(fuzzy_similarity_matrix_2)[0][-5:][::-1]
    #     return (
    #         appended_comments,
    #         [
    #             (
    #                 self.license_dataset_df.loc[idx, 'License Name'],
    #                 fuzzy_similarity_matrix_2[0][idx]
    #             ) 
    #             for idx in top_5_indices
    #         ]
    #     )

    def scan(self, filePath): 
        '''
        Scans a file for potential licenses using semantic search and fuzzy string matching.

        :param processedData: Processed Data form input file
        :return: Returns possible licenses contained in the input file based
                on matching using semantic search
        '''

        file_comments = self.extract_comments(filePath)
        chars_to_remove = ['—', '…', '•', '§', '«', '»', '„', '・', '−', '*', '>', '<']
        for char_to_remove in chars_to_remove:
            file_comments = file_comments.replace(char_to_remove, '')

        file_comments = file_comments.split('\n')

        license_index_map = {}
        all_license_texts = []
        for license_index, license_text in enumerate(self.license_dataset_df['License Text']):
            for line in license_text.split('\n'):
                all_license_texts.append(line)
                license_index_map[len(all_license_texts) - 1] = license_index
        
        results = []
        fuzzy_similarity_matrix = np.zeros((len(file_comments), len(all_license_texts)))
        for index, comment in enumerate(file_comments):
            for i in range(len(all_license_texts)):
                fuzzy_similarity_matrix[index][i] = fuzz.ratio(comment, all_license_texts[i]) 
            max_score_index = np.argmax(fuzzy_similarity_matrix[index]) 
            results.append(
                        (
                            fuzzy_similarity_matrix[index][max_score_index], 
                            comment
                        )
                    ) 

        appended_comments = []
        for result in results:
            if result[0] >= 30:
                appended_comments.append(result[1])
            else:
                break
        
        appended_comments = "\n".join(appended_comments)
        fuzzy_similarity_matrix_2 = np.zeros(len(self.license_dataset_df))
        for i in range(len(self.license_dataset_df)):
            fuzzy_similarity_matrix_2[i] = fuzz.ratio(appended_comments, self.license_dataset_df.loc[i, 'License Text'])
        top_5_indices = np.argsort(fuzzy_similarity_matrix_2)[-5:][::-1]
        return ({
          "shortname": self.license_dataset_df.loc[top_5_indices[0], 'License ID'],
          "sim_score": fuzzy_similarity_matrix_2[top_5_indices[0]],
          "sim_type": "SemanticSearch-LVD",
          "description": ""
        })

        # appended_comments = "\n".join(appended_comments)
        # fuzzy_similarity_matrix_2 = np.zeros(len(self.license_dataset_df))
        # for i in range(len(self.license_dataset_df)):
        #     fuzzy_similarity_matrix_2[i] = fuzz.ratio(appended_comments, self.license_dataset_df.loc[i, 'License Text'])
        # top_5_indices = np.argsort(fuzzy_similarity_matrix_2)[-5:][::-1]
        # return (
        #     appended_comments,
        #     [
        #         (
        #             self.license_dataset_df.loc[idx, 'License Name'],
        #             fuzzy_similarity_matrix_2[idx]
        #         ) 
        #         for idx in top_5_indices
        #     ]
        # )

In [43]:
agent = SemanticSearchAgent()

In [44]:
test_file_path = '/home/jimbo/Desktop/GSoC24/repo/atarashi/atarashi/evaluator/TestFiles'
test_file_columns = ['file path', 'licenses']

test_files = os.listdir(test_file_path)

test_df_atarashi = pd.DataFrame(columns=test_file_columns)

for file in test_files:
    temp_df = pd.DataFrame({'file path': os.path.join(test_file_path, file), 'licenses': [file.split('.')[0]]})
    test_df_atarashi = pd.concat([test_df_atarashi, temp_df], ignore_index=True)

for index, row in test_df_atarashi.iterrows():
    try:
        with open(row['file path'], "r", encoding='utf-8') as f:
            comments = f.read()
    except:
        print(f'Dropping Index: {index}')
        test_df_atarashi = test_df_atarashi.drop(index)
test_df_atarashi = test_df_atarashi.reset_index()

In [45]:
test_df_atarashi.loc[0, 'file path']

'/home/jimbo/Desktop/GSoC24/repo/atarashi/atarashi/evaluator/TestFiles/BitTorrent-1.1.py'

In [46]:
os.path.exists(test_df_atarashi.loc[0, 'file path'])

True

In [ ]:
from tqdm import tqdm

for index, row in tqdm(test_df_atarashi.iterrows()):
    result = agent.scan(row['file path'])
    test_df_atarashi.loc[index, 'result'] = result['shortname']

In [4]:
test_file_path = 'extras/LastGoodNomosTestfilesScan.txt'
test_file_columns = ['file path', 'licenses']

with open(test_file_path, 'r') as file:
    test_data = file.readlines()

test_df = pd.DataFrame(columns=test_file_columns)

for line in test_data:
    if "contains license(s)" in line:  # Check for the expected pattern
        licenses = line.split("contains license(s) ")[1].strip().split("\\")[0]
        file_path = line.split('File ')[1].split(' contains license(s)')[0].strip()
        licenses = str(licenses)
        licenses = licenses.split(',')
        licenses = '\n'.join(licenses)
        temp_df = pd.DataFrame({'file path': file_path, 'licenses': [licenses]})
        test_df = pd.concat([test_df, temp_df], ignore_index=True)

for index, row in test_df.iterrows():
    try:
        with open(os.path.join('extras', row['file path']), "r", encoding='utf-8') as f:
            comments = f.read()
    except:
        print(f'Dropping Index: {index}')
        test_df = test_df.drop(index)
test_df = test_df.reset_index()

Dropping Index: 39
Dropping Index: 83
Dropping Index: 160
Dropping Index: 161
Dropping Index: 162
Dropping Index: 202
Dropping Index: 301
Dropping Index: 322
Dropping Index: 350
Dropping Index: 548
Dropping Index: 625
Dropping Index: 646
Dropping Index: 775
Dropping Index: 888
Dropping Index: 902
Dropping Index: 916
Dropping Index: 939
Dropping Index: 940
Dropping Index: 963
Dropping Index: 1394
Dropping Index: 1398
Dropping Index: 1399
Dropping Index: 1439
Dropping Index: 2009


In [7]:
test_df.loc[4]

index                                          4
file path    NomosTestfiles/ACE/ACE-copying.html
licenses                                     ACE
Name: 4, dtype: object

In [8]:
agent.license_dataset_df[agent.license_dataset_df['License ID'] == 'ACE']

,Unnamed: 0,License Name,License ID,License Text


In [18]:
agent.scan(os.path.join('extras', test_df.loc[0, 'file path']))

{'shortname': 'AAL',
 'sim_score': 96.0,
 'sim_type': 'SemanticSearch-LVD',
 'description': ''}

In [ ]:
# agent.license_dataset_df[agent.license_dataset_df['License ID'] == 'AAL']
print(agent.license_dataset_df.loc[624, 'License Text'])